In [0]:
df_emp = spark.read.csv("/Volumes/dev/bronze/transactions/transactions.csv/", header=True, inferSchema=True)
df_emp.display()


In [0]:
df_patients = spark.read.csv("/Volumes/dev/bronze/transactions/patients.csv/", header=True, inferSchema=True)
df_patients .display()

In [0]:
from pyspark.sql.functions import regexp_replace, sha2, col, lit, substring, concat

df_mask = df_patients.withColumn("ssn_masked", regexp_replace("SSN", "\\d", "X")).withColumn("phone_masked", concat(lit("XXX-XXX-"), substring(col("PhoneNumber"), -4, 4))).withColumn("Address_New", sha2(col("Address"),256))
     


df_mask.display()

In [0]:
df_mask = df_patients.withColumn("ssn_masked", concat(lit("XXXX-XXXX-"), substring(col("SSN"), -4, 4))).withColumn("phone_masked", concat(lit("XXX-XXX-"), substring(col("PhoneNumber"), -4, 4))).withColumn("Address_New", sha2(col("Address"),256))

df_mask.display()

In [0]:
%sql
SELECT 
    CONCAT('XXX-XX-', RIGHT(ssn, 4)) AS ssn_masked,
    HASHBYTES('SHA2_256', email) AS email_hash,
    CONCAT('XXX', RIGHT(phone, 4)) AS phone_masked
FROM member_table;

In [0]:
%sql
ALTER TABLE member_table
ALTER COLUMN email ADD MASKED WITH (FUNCTION = 'email()');

In [0]:
Derived Column:
    SSN_masked = mask(SSN, 'X')
    Email_masked = sha2(Email, 256)
    Phone_masked = concat('XXX-XXX-', right(Phone, 4))